<a href="https://colab.research.google.com/github/sheng13/Ai--/blob/main/%E3%80%8C0704_Colab_LINE_Bot_with_GEMINI_Tooluse_copy_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [17]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [18]:
import os
from pyngrok import ngrok

In [19]:
ngrok.kill()

In [20]:
import requests

ngrok.kill() # Add this line to kill any existing tunnels
ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://alla-mammillate-tawanna.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://alla-mammillate-tawanna.ngrok-free.dev


True

In [21]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [22]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [23]:
result = stateful_query("iphone歷史")
print(result)

iPhone 的歷史是一部創新與變革的編年史，它徹底改變了智慧型手機市場，並深刻影響了人們的日常生活。

**誕生與初期發展 (2007-2010)**

2007 年 1 月 9 日，時任蘋果公司執行長史蒂夫·賈伯斯（Steve Jobs）在 Macworld Expo 大會上發表了第一代 iPhone，這款產品結合了寬螢幕 iPod、革命性的行動電話和突破性的網路通訊設備的功能。同年 6 月 29 日，初代 iPhone 正式發售。它以其直觀的多點觸控螢幕、虛擬鍵盤和簡約設計，開創了觸控式手機時代，並被《時代雜誌》評選為「2007 年的年度發明」。

隨後，蘋果公司不斷推出新機型，逐步完善 iPhone 的功能：
*   **iPhone 3G (2008)**：增加了 3G 網路連接功能，並推出了 App Store，為第三方應用程式開啟了大門，極大地擴展了 iPhone 的應用潛力。
*   **iPhone 3GS (2009)**：提升了處理速度，並首次支援視訊錄製和語音控制功能。
*   **iPhone 4 (2010)**：引入了 Retina 顯示器，大幅提升了螢幕解析度，並首次配備前置鏡頭，實現了 FaceTime 視訊通話。

**功能與設計的演進 (2011-2016)**

在這段時期，iPhone 在硬體和軟體方面都取得了顯著進步：
*   **iPhone 4S (2011)**：首次搭載了智慧語音助理 Siri，並提升了相機像素和錄影功能。
*   **iPhone 5 (2012)**：螢幕尺寸首次從 3.5 吋擴大到 4 吋，並採用了全新的 Lightning 連接器。
*   **iPhone 5S 和 iPhone 5C (2013)**：iPhone 5S 首次引入了 Touch ID 指紋辨識功能，而 iPhone 5C 則提供了多種鮮豔的顏色選擇，定位更加親民。
*   **iPhone 6 和 iPhone 6 Plus (2014)**：螢幕尺寸進一步增大，分別達到 4.7 吋和 5.5 吋，滿足了用戶對大螢幕的需求。
*   **iPhone 6s 和 iPhone 6s Plus (2015)**：引入了 3D Touch 壓感觸控技術，並將主相機像素提升至 1200 萬，支援 4K 影片錄製。
*   **i

In [24]:
result2 = stateful_query("iphone創始人")
print(result2)

None


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 23:06:29] "POST / HTTP/1.1" 400 -


BODY:  {"destination":"U287a3d06a20703e880b3c1fda99f4939","events":[]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 23:06:56] "POST / HTTP/1.1" 400 -


BODY:  {"destination":"U287a3d06a20703e880b3c1fda99f4939","events":[{"type":"message","message":{"type":"text","id":"595584309441331643","quoteToken":"O7aH8FefmRBZShIiLn1fpMeq08Uci5ej4TACFgxZHchsOB5lHIiWzRvUGkIWnMG8LQTST6hxxMKjZ54HoG7tEsyI6xKmpAPQFvN4IM3YSemY0nO1KGuKEcGVUOqF3NzBJVntRaOeUCXRI9vVUOgaRQ","markAsReadToken":"DEzY3JVKo26M2zEWIGFMiHwnAL1fOhEPn6h-yFKjhIthIWQltd06HsL7kDpOL4wpRBILTDxh8-QThEX4VccDVBwi3zyRhIgrZtqZp1F_H_hPPxF-8Q1AaDUnUtEarTeH0NL51RnW0nmiTXoHv1OgSGLmYdYgm4jdNOE5wXMc1W4nqq3uL-dg527nTDEh8hV8LwSScPVbLOlnZWT6HTfuVg","text":"AIiphone歷史"},"webhookEventId":"01KEDBD63QYMJDWNBRMKK9C68E","deliveryContext":{"isRedelivery":false},"timestamp":1767827215992,"source":{"type":"user","userId":"Ubffffa3746b3dc784a8659f16c5692d8"},"replyToken":"689f075cbdce431dbfcf8a4bd79f0e92","mode":"active"}]}
